In [1]:
from structured.generator import StructuredGeneration
from src.inference.model import load_model
from structured.schemas import TicketOutput
from prompts.classifier import build_ticket_classifier
from structured.parser import parse_and_validate_ticket

In [2]:
import pandas as pd 

df = pd.read_csv('data/tickets.csv')

In [3]:
tokenizer, model = load_model()
generator = StructuredGeneration(model, tokenizer)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [6]:
# ticket_id,message,category,sentiment,urgency

def evalute_ticket(generator, row ) :

    prompt = build_ticket_classifier(row['message'])
    
    try : 
        result = generator.generate(prompt, TicketOutput)
        result = parse_and_validate_ticket(result)
        return {
            'ticket_id' : row['ticket_id'],
            'result'    : result,
            'expected_category' : row['category'],
            'expected_sentiment': row['sentiment'],
            'expected_urgency'  : row['urgency'],
            'success'   : True,
            'error'     : None
        }

    except Exception as e :
        return {
            'ticket_id' : row['ticket_id'],
            'result'    : None,
            'expected_category' : row['category'],
            'expected_sentiment': row['sentiment'],
            'expected_urgency'  : row['urgency'],
            'success'   : False,
            'error'     : str(e)
        }
    

In [7]:
results = []
for _, row in df.iterrows() : 
    output = evalute_ticket(generator, row)
    results.append(output)


In [8]:
results[0]

{'ticket_id': 1,
 'result': TicketOutput(category='technical', sentiment='positive', urgency='high', summary='I need urgent help with my laptop screen.'),
 'expected_category': 'technical',
 'expected_sentiment': 'negative',
 'expected_urgency': 'high',
 'success': True,
 'error': None}

In [9]:
validity_rate = sum([r['success'] for r in results]) / len(results)
print(f"{validity_rate:.2%}")

100.00%


In [10]:
def check_category(results) :
    if results['success'] is False : 
        return False
    return (results['expected_category'] == results['result'].category )

In [11]:
category_accuracy = sum([check_category(r) for r in results])/len(results)
print(f"{category_accuracy:.2%}")

40.00%


In [12]:
def check_sentiment(results) :
    if results['success'] is False : 
        return False
    return (results['expected_sentiment'] == results['result'].sentiment )

sentiment_accuracy = sum([check_sentiment(r) for r in results])/len(results)
print(f"{sentiment_accuracy:.2%}")

40.00%


In [13]:
def check_urgency(results) :
    if results['success'] is False : 
        return False
    return (results['expected_urgency'] == results['result'].urgency )

urgency_accuracy = sum([check_urgency(r) for r in results])/len(results)
print(f"{urgency_accuracy:.2%}")

50.00%


In [14]:
def is_successful(result) : 
    if result['success'] is False : 
        return False

    output= result['result']

    return(
        result['expected_category'] == output.category
        and 
        result['expected_urgency'] == output.urgency
        and 
        result['expected_sentiment'] == output.sentiment
    )


In [15]:
[is_successful(r) for r in results]

[False, False, False, False, False, False, False, True, False, False]

In [16]:
def build_evaluation_row(result):

    row = {
        "ticket_id": result["ticket_id"],
        "success": result["success"],
    }

    # Generation failed
    if not result["success"]:
        row.update({
            "expected_category": result["expected_category"],
            "predicted_category": None,
            "category_correct": False,

            "expected_sentiment": result["expected_sentiment"],
            "predicted_sentiment": None,
            "sentiment_correct": False,

            "expected_urgency": result["expected_urgency"],
            "predicted_urgency": None,
            "urgency_correct": False,

            "error": result["error"],
        })

        return row

    # Generation succeeded
    output = result["result"]

    row.update({
        "expected_category": result["expected_category"],
        "predicted_category": output.category,
        "category_correct": (
            output.category == result["expected_category"]
        ),

        "expected_sentiment": result["expected_sentiment"],
        "predicted_sentiment": output.sentiment,
        "sentiment_correct": (
            output.sentiment == result["expected_sentiment"]
        ),

        "expected_urgency": result["expected_urgency"],
        "predicted_urgency": output.urgency,
        "urgency_correct": (
            output.urgency == result["expected_urgency"]
        ),

        "error": None,
    })

    return row

In [17]:
evaluation_rows = [
    build_evaluation_row(result)
    for result in results
]

evaluation_table = pd.DataFrame(evaluation_rows)
evaluation_table

,ticket_id,success,expected_category,predicted_category,category_correct,expected_sentiment,predicted_sentiment,sentiment_correct,expected_urgency,predicted_urgency,urgency_correct,error
0,1,True,technical,technical,True,negative,positive,False,high,high,True,None
1,2,True,account,delivery,False,neutral,positive,False,medium,high,False,None
2,3,True,delivery,delivery,True,negative,negative,True,medium,high,False,None
3,4,True,billing,delivery,False,negative,positive,False,high,high,True,None
4,5,True,subscription,delivery,False,neutral,positive,False,medium,low,False,None
5,6,True,account,delivery,False,positive,positive,True,low,low,True,None
6,7,True,billing,account,False,negative,positive,False,high,high,True,None
7,8,True,delivery,delivery,True,positive,positive,True,low,low,True,None
8,9,True,technical,technical,True,negative,negative,True,high,medium,False,None
9,10,True,subscription,technical,False,neutral,positive,False,low,medium,False,None


In [18]:
evaluation_table[evaluation_table['category_correct']==False]

,ticket_id,success,expected_category,predicted_category,category_correct,expected_sentiment,predicted_sentiment,sentiment_correct,expected_urgency,predicted_urgency,urgency_correct,error
1,2,True,account,delivery,False,neutral,positive,False,medium,high,False,None
3,4,True,billing,delivery,False,negative,positive,False,high,high,True,None
4,5,True,subscription,delivery,False,neutral,positive,False,medium,low,False,None
5,6,True,account,delivery,False,positive,positive,True,low,low,True,None
6,7,True,billing,account,False,negative,positive,False,high,high,True,None
9,10,True,subscription,technical,False,neutral,positive,False,low,medium,False,None


In [4]:
prompt = build_ticket_classifier(df.iloc[2]['message'])
generator.generate(prompt,TicketOutput)

'{ "category": "delivery", "sentiment": "negative", "urgency": "high" , "summary": "The customer package arrived two days late. The delivery time was 2 days, and the delivery was late." }'

In [ ]:
from evaluation.metrics import sentiment_accuracy

from evaluation import sentiment_accuracy

In [5]:
print(prompt)

Role:
            You are a precise ticket classification assistant.

            Constraints:
            - Return only the requested result.
- Do not provide explanations.
- Do not use markdown.
- summary must be one sentence as max

            Task:
            Classify the customer ticket.

            Input:
            <input>
            My package arrived two days late.
            </input>


In [22]:
# ============================================================
# 1. Imports
# ============================================================
import os 
import sys

sys.path.append(os.path.abspath('..'))


import pandas as pd

from evaluation import (
    TicketEvaluator,
    validity_rate,
    category_accuracy,
    sentiment_accuracy,
    urgency_accuracy,
    overall_success_rate,
    failure_count,
)
# ============================================================
# 2. Load evaluation dataset
# ============================================================

df = pd.read_csv("../data/tickets.csv")

df.head()
# ============================================================
# 3. Create evaluator
# ============================================================

evaluator = TicketEvaluator(generator)
# ============================================================
# 4. Run evaluation
# ============================================================

results = evaluator.evaluate(df)

print(f"Evaluated {len(results)} tickets")
# ============================================================
# 5. Calculate metrics
# ============================================================

validity = validity_rate(results)
category_acc = category_accuracy(results)
sentiment_acc = sentiment_accuracy(results)
urgency_acc = urgency_accuracy(results)
overall_success = overall_success_rate(results)
failures = failure_count(results)

print(f"Validity Rate:       {validity:.2%}")
print(f"Category Accuracy:   {category_acc:.2%}")
print(f"Sentiment Accuracy:  {sentiment_acc:.2%}")
print(f"Urgency Accuracy:    {urgency_acc:.2%}")
print(f"Overall Success:     {overall_success:.2%}")
print(f"Generation Failures: {failures}")
# ============================================================
# 6. Inspect individual results
# ============================================================

for result in results:
    print(result)
# ============================================================
# 7. Inspect generation failures
# ============================================================

generation_failures = [
    result
    for result in results
    if not result.success
]

for result in generation_failures:
    print(f"Ticket {result.ticket_id}")
    print(f"Error: {result.error}")
    print("-" * 50)
# ============================================================
# 8. Inspect classification failures
# ============================================================

classification_failures = [
    result
    for result in results
    if result.success
    and (
        not result.category_correct
        or not result.sentiment_correct
        or not result.urgency_correct
    )
]

for result in classification_failures:

    print(f"Ticket {result.ticket_id}")

    if not result.category_correct:
        print(
            f"  Category: "
            f"expected={result.expected_category}, "
            f"predicted={result.result.category}"
        )

    if not result.sentiment_correct:
        print(
            f"  Sentiment: "
            f"expected={result.expected_sentiment}, "
            f"predicted={result.result.sentiment}"
        )

    if not result.urgency_correct:
        print(
            f"  Urgency: "
            f"expected={result.expected_urgency}, "
            f"predicted={result.result.urgency}"
        )

    print("-" * 50)
# ============================================================
# 9. Create a simple evaluation table
# ============================================================

evaluation_table = pd.DataFrame([
    {
        "ticket_id": r.ticket_id,

        "expected_category": r.expected_category,
        "predicted_category": (
            r.result.category
            if r.result else None
        ),
        "category_correct": r.category_correct,

        "expected_sentiment": r.expected_sentiment,
        "predicted_sentiment": (
            r.result.sentiment
            if r.result else None
        ),
        "sentiment_correct": r.sentiment_correct,

        "expected_urgency": r.expected_urgency,
        "predicted_urgency": (
            r.result.urgency
            if r.result else None
        ),
        "urgency_correct": r.urgency_correct,

        "success": r.success,
        "error": r.error,
    }
    for r in results
])

evaluation_table
# ============================================================
# 10. Show only failed cases
# ============================================================

failed_cases = evaluation_table[
    ~(
        evaluation_table["category_correct"]
        & evaluation_table["sentiment_correct"]
        & evaluation_table["urgency_correct"]
    )
]

failed_cases

ModuleNotFoundError: No module named 'evaluation'

In [4]:
!pip install outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00


In [1]:
import pandas as pd 
from evaluation import (
    TicketEvaluator,
    validity_rate,
    category_accuracy,
    sentiment_accuracy,
    urgency_accuracy,
    overall_success_rate,
    failure_count
)

In [2]:
from structured.generator import StructuredGeneration
from src.inference.model import load_model


tokenizer, model = load_model()
generator   = StructuredGeneration(model, tokenizer)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
DATA_PATH = 'data/tickets.csv'
df = pd.read_csv(DATA_PATH)
print(len(df))

10


In [4]:
evaluator = TicketEvaluator(generator)

In [5]:
results = evaluator.evaluate(df)

RAW RESULT: { "category": "technical", "sentiment": "positive", "urgency": "high", "summary": "I need urgent help with my laptop screen." }
RAW TYPE: <class 'str'>
PARSED RESULT: category='technical' sentiment='positive' urgency='high' summary='I need urgent help with my laptop screen.'
PARSED TYPE: <class 'structured.schemas.TicketOutput'>
RAW RESULT: { "category": "delivery", "sentiment": "positive", "urgency": "high", "summary": "I forgot my password and need to reset it. I need to reset it now." }
RAW TYPE: <class 'str'>
PARSED RESULT: category='delivery' sentiment='positive' urgency='high' summary='I forgot my password and need to reset it. I need to reset it now.'
PARSED TYPE: <class 'structured.schemas.TicketOutput'>
RAW RESULT: { "category": "delivery", "sentiment": "negative", "urgency": "high" , "summary": "The customer package arrived two days late. The delivery time was 2 days, and the delivery was late." }
RAW TYPE: <class 'str'>
PARSED RESULT: category='delivery' sentimen

In [6]:
metrics = { "Total Tickets": len(results), "Validity Rate": validity_rate(results), "Category Accuracy": category_accuracy(results), "Sentiment Accuracy": sentiment_accuracy(results), "Urgency Accuracy": urgency_accuracy(results), "Overall Success Rate": overall_success_rate(results), "Generation Failures": failure_count(results), }

In [7]:
print("=" * 60)
print(" EVALUATION REPORT") 
print("=" * 60) 
print(f"Total Tickets: {metrics['Total Tickets']}") 
print( f"Validity Rate: " f"{metrics['Validity Rate']:.2%}" )
print( f"Category Accuracy: " f"{metrics['Category Accuracy']:.2%}" ) 
print( f"Sentiment Accuracy: " f"{metrics['Sentiment Accuracy']:.2%}" )
print( f"Urgency Accuracy: " f"{metrics['Urgency Accuracy']:.2%}" ) 
print( f"Overall Success: " f"{metrics['Overall Success Rate']:.2%}" )
print( f"Generation Failures: " f"{metrics['Generation Failures']}" ) 
print("=" * 60)

 EVALUATION REPORT
Total Tickets: 10
Validity Rate: 100.00%
Category Accuracy: 40.00%
Sentiment Accuracy: 50.00%
Urgency Accuracy: 50.00%
Overall Success: 10.00%
Generation Failures: 0
